# CompVP Part 1 — high-resolution rebuild

Hadis Nouri · individual project. All three tasks of Part 1. This notebook executes all model inference and rebuilds the exact submission ZIP and presentation from pinned source inputs. No tokens are stored in this notebook.

The student reviewed and confirmed the ten-pair subset. Actual local GUI/source screenshots are supplied as presentation assets. The dataset and model attachments allow an offline execution.


In [ ]:
from pathlib import Path
import os,sys,json,zipfile,subprocess,shutil
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
inputs=Path('/kaggle/input')
archives=[]; expanded=[]
for directory,_,names in os.walk(inputs,followlinks=True):
    base=Path(directory)
    if 'project_input.zip' in names: archives.append(base/'project_input.zip')
    if 'reproduce_project.py' in names and base.name=='src': expanded.append(base.parent)
project=Path('/kaggle/working/project_input')
if archives:
    assert len(archives)==1, archives
    project.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        assert z.testzip() is None
        z.extractall(project)
else:
    assert len(expanded)==1, {'archives':archives,'expanded':expanded,'mounts':list(inputs.iterdir())}
    shutil.copytree(expanded[0],project,dirs_exist_ok=True)
for source in (project/'src').glob('*.py'):
    compile(source.read_text(),str(source),'exec')
wheels=list((project/'wheels').glob('*.whl'))
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',*map(str,wheels)],check=True)
print('Reproducibility input archive extracted; PowerPoint dependencies available.')


## Verify attached official models and run the complete project
The runner independently verifies model/config checksums before inference. All 100 originals and all 100 Mooneys are processed by each system.


In [ ]:
config_dirs=[]
for directory,_,names in os.walk(inputs,followlinks=True):
    if 'config.json' in names:
        p=Path(directory)
        try: typ=json.loads((p/'config.json').read_text()).get('model_type')
        except (OSError,ValueError): continue
        config_dirs.append((p,typ))
depth_dirs=[p for p,t in config_dirs if t=='depth_anything' and (p/'model.safetensors').exists()]
clip_dirs=[p for p,t in config_dirs if t=='clip' and ((p/'pytorch_model.bin').exists() or (p/'model.safetensors').exists())]
assert len(depth_dirs)==len(clip_dirs)==1,(depth_dirs,clip_dirs)
out=Path('/kaggle/working/part1_rebuild')
command=[sys.executable,str(project/'src/reproduce_project.py'),'--project-root',str(project),'--output',str(out),'--original',str(project/'Original'),'--depth-model',str(depth_dirs[0]),'--clip-model',str(clip_dirs[0])]
result=subprocess.run(command,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
Path('/kaggle/working/rebuild_execution.log').write_text(result.stdout)
print(result.stdout)
assert result.returncode==0, 'See rebuild_execution.log and part1_rebuild/analysis logs'


## Results and downloadable submission
Download `CompVP_Part1_Hadis_Nouri.zip` from the Output tab for the course submission. The source/analysis ZIP is a separate reproducibility artifact. Depth values are relative inverse depth; CLIP shares depend on the eight prompts. The optional 8AFC test is same-photo matching, not Paper 1's cross-pose human experiment.


In [ ]:
for name in ['CompVP_Part1_Hadis_Nouri.zip','CompVP_Part1_Source_Analysis.zip']:
    shutil.copyfile(out/name,Path('/kaggle/working')/name)
for name in ['run_info.json','summary.json','verification_results.json','archive_checksums.json']:
    shutil.copyfile(out/'analysis'/name,Path('/kaggle/working')/name)
shutil.copyfile(out/'submission/CompVP_Part1_Hadis_Nouri.pptx',Path('/kaggle/working/CompVP_Part1_Hadis_Nouri.pptx'))
print(json.dumps(json.loads((out/'analysis/summary.json').read_text()),indent=2))
from IPython.display import display,Image as DisplayImage,FileLink
for name in ['depth_statistics.png','clip_statistics.png','afc8_statistics.png']:
    display(DisplayImage(filename=str(out/'figures'/name)))
display(FileLink('/kaggle/working/CompVP_Part1_Hadis_Nouri.zip'))
display(FileLink('/kaggle/working/CompVP_Part1_Source_Analysis.zip'))
